# Advanced Data Loading for LLMs

## Historical Context

Data loading has evolved with model scale:

- **2012-2017**: Simple mini-batch loading, images fit in RAM
- **2018-2019**: BERT/GPT-2: variable-length sequences, basic padding
- **2020**: GPT-3 scale: Need efficient packing, streaming from disk
- **2021**: Datasets library (HuggingFace): Memory-mapped files, streaming
- **2022**: Multi-trillion token datasets, need for sequence packing
- **2023**: LLaMA/Mistral: Dynamic batching, length-based bucketing
- **2024-2025**: Petabyte-scale streaming, on-the-fly preprocessing

## Why Advanced Loading Matters

**Efficiency gains**:
- **Sequence packing**: 2-3x throughput by eliminating padding waste
- **Dynamic batching**: Better GPU utilization
- **Streaming**: Train on datasets larger than disk/RAM
- **Prefetching**: Hide data loading latency behind compute

**Modern LLM training** on trillion-token datasets requires:
1. Streaming from cloud storage
2. Efficient packing to minimize padding
3. Multi-worker prefetching
4. Memory-mapped files for fast random access

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, IterableDataset
import numpy as np
import time
import matplotlib.pyplot as plt
from typing import List, Dict, Iterator
import random
from collections import defaultdict

## 1. Sequence Packing for Efficiency

**Problem**: Variable-length sequences waste compute on padding
- Example: max_len=2048, but average length=512
- 75% of tokens are padding = 75% wasted compute

**Solution**: Pack multiple short sequences into one training example

In [ ]:
class SequencePacker:
    """Efficiently pack multiple sequences into fixed-length chunks.
    
    Used in modern LLM training to maximize GPU utilization.
    """
    
    def __init__(self, max_length: int, pad_token_id: int = 0):
        self.max_length = max_length
        self.pad_token_id = pad_token_id
        
    def pack_sequences(self, sequences: List[List[int]]) -> List[Dict[str, torch.Tensor]]:
        """Pack sequences using greedy algorithm.
        
        Returns:
            List of packed samples with input_ids and attention_mask
        """
        packed_samples = []
        current_batch = []
        current_length = 0
        
        for seq in sequences:
            seq_len = len(seq)
            
            # If adding this sequence would exceed max_length, finalize current batch
            if current_length + seq_len > self.max_length:
                if current_batch:
                    packed_samples.append(self._finalize_batch(current_batch, current_length))
                current_batch = [seq]
                current_length = seq_len
            else:
                current_batch.append(seq)
                current_length += seq_len
        
        # Finalize last batch
        if current_batch:
            packed_samples.append(self._finalize_batch(current_batch, current_length))
        
        return packed_samples
    
    def _finalize_batch(self, sequences: List[List[int]], total_length: int) -> Dict[str, torch.Tensor]:
        """Concatenate sequences and create attention mask."""
        # Concatenate all sequences
        input_ids = []
        for seq in sequences:
            input_ids.extend(seq)
        
        # Pad to max_length
        padding_length = self.max_length - len(input_ids)
        input_ids.extend([self.pad_token_id] * padding_length)
        
        # Create attention mask (1 for real tokens, 0 for padding)
        attention_mask = [1] * total_length + [0] * padding_length
        
        # Create position IDs that reset for each packed sequence
        position_ids = []
        for seq in sequences:
            position_ids.extend(range(len(seq)))
        position_ids.extend([0] * padding_length)
        
        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'position_ids': torch.tensor(position_ids, dtype=torch.long),
        }


# Demonstrate efficiency gain
def compare_packing_efficiency():
    """Compare naive padding vs sequence packing."""
    
    # Generate variable-length sequences
    np.random.seed(42)
    sequences = [
        list(range(length)) 
        for length in np.random.randint(100, 800, size=100)
    ]
    
    max_length = 2048
    packer = SequencePacker(max_length)
    
    # Naive approach: pad each sequence to max_length
    naive_tokens = len(sequences) * max_length
    real_tokens = sum(len(seq) for seq in sequences)
    
    # Packed approach
    packed_samples = packer.pack_sequences(sequences)
    packed_tokens = len(packed_samples) * max_length
    
    print(f"Sequence Packing Efficiency Analysis")
    print(f"{'='*60}")
    print(f"Number of sequences: {len(sequences)}")
    print(f"Real tokens: {real_tokens:,}")
    print(f"\nNaive Padding:")
    print(f"  Total tokens: {naive_tokens:,}")
    print(f"  Padding waste: {(naive_tokens - real_tokens) / naive_tokens * 100:.1f}%")
    print(f"  Batches needed: {len(sequences)}")
    print(f"\nSequence Packing:")
    print(f"  Total tokens: {packed_tokens:,}")
    print(f"  Padding waste: {(packed_tokens - real_tokens) / packed_tokens * 100:.1f}%")
    print(f"  Batches needed: {len(packed_samples)}")
    print(f"\nSpeedup: {naive_tokens / packed_tokens:.2f}x")
    
    # Visualize packing
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Show first few sequences with naive padding
    naive_data = np.zeros((min(10, len(sequences)), max_length))
    for i, seq in enumerate(sequences[:10]):
        naive_data[i, :len(seq)] = 1
    
    ax1.imshow(naive_data, aspect='auto', cmap='Blues', interpolation='nearest')
    ax1.set_xlabel('Token Position', fontsize=12)
    ax1.set_ylabel('Sequence', fontsize=12)
    ax1.set_title('Naive Padding (blue=real, white=padding)', fontsize=14, fontweight='bold')
    
    # Show packed sequences
    packed_data = np.zeros((min(10, len(packed_samples)), max_length))
    for i, sample in enumerate(packed_samples[:10]):
        mask = sample['attention_mask'].numpy()
        packed_data[i] = mask
    
    ax2.imshow(packed_data, aspect='auto', cmap='Blues', interpolation='nearest')
    ax2.set_xlabel('Token Position', fontsize=12)
    ax2.set_ylabel('Packed Batch', fontsize=12)
    ax2.set_title('Sequence Packing (much less white!)', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('sequence_packing_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

compare_packing_efficiency()

## 2. Dynamic Batching by Length

**Problem**: Fixed batch size wastes memory
- Long sequences: underfill GPU
- Short sequences: could fit more

**Solution**: Adjust batch size based on sequence length

In [ ]:
class DynamicBatchSampler:
    """Create batches with dynamic size based on sequence length.
    
    Maximizes GPU utilization by fitting more short sequences,
    fewer long sequences per batch.
    """
    
    def __init__(self, lengths: List[int], max_tokens: int, 
                 shuffle: bool = True, drop_last: bool = False):
        self.lengths = lengths
        self.max_tokens = max_tokens
        self.shuffle = shuffle
        self.drop_last = drop_last
        
    def __iter__(self) -> Iterator[List[int]]:
        """Generate batches of indices."""
        indices = list(range(len(self.lengths)))
        
        if self.shuffle:
            random.shuffle(indices)
        
        # Sort by length for efficient batching
        indices.sort(key=lambda i: self.lengths[i])
        
        batch = []
        batch_tokens = 0
        
        for idx in indices:
            length = self.lengths[idx]
            
            # Check if adding this sample would exceed max_tokens
            new_batch_tokens = max(batch_tokens, length) * (len(batch) + 1)
            
            if new_batch_tokens > self.max_tokens and batch:
                yield batch
                batch = [idx]
                batch_tokens = length
            else:
                batch.append(idx)
                batch_tokens = max(batch_tokens, length)
        
        if batch and not self.drop_last:
            yield batch
    
    def __len__(self) -> int:
        # Approximate
        return len(self.lengths) // (self.max_tokens // max(self.lengths))


class LengthBucketSampler:
    """Bucket sequences by length, then sample from buckets.
    
    Reduces padding within batches while maintaining randomness.
    """
    
    def __init__(self, lengths: List[int], batch_size: int, 
                 bucket_boundaries: List[int]):
        self.lengths = lengths
        self.batch_size = batch_size
        self.bucket_boundaries = sorted(bucket_boundaries)
        
        # Assign sequences to buckets
        self.buckets = defaultdict(list)
        for idx, length in enumerate(lengths):
            bucket_id = self._get_bucket(length)
            self.buckets[bucket_id].append(idx)
    
    def _get_bucket(self, length: int) -> int:
        """Find bucket for given length."""
        for i, boundary in enumerate(self.bucket_boundaries):
            if length <= boundary:
                return i
        return len(self.bucket_boundaries)
    
    def __iter__(self) -> Iterator[List[int]]:
        """Sample batches from buckets."""
        # Shuffle within each bucket
        for indices in self.buckets.values():
            random.shuffle(indices)
        
        # Create batches from each bucket
        all_batches = []
        for indices in self.buckets.values():
            for i in range(0, len(indices), self.batch_size):
                batch = indices[i:i + self.batch_size]
                if len(batch) == self.batch_size:  # Drop incomplete batches
                    all_batches.append(batch)
        
        # Shuffle batches for randomness
        random.shuffle(all_batches)
        
        for batch in all_batches:
            yield batch
    
    def __len__(self) -> int:
        return sum(len(indices) // self.batch_size for indices in self.buckets.values())


# Benchmark dynamic batching
def benchmark_dynamic_batching():
    """Compare fixed vs dynamic batch sizing."""
    
    # Generate variable-length sequences
    np.random.seed(42)
    lengths = np.random.randint(128, 2048, size=1000).tolist()
    
    # Fixed batch size
    fixed_batch_size = 32
    fixed_batches = len(lengths) // fixed_batch_size
    fixed_max_length = max(lengths)
    fixed_total_tokens = fixed_batches * fixed_batch_size * fixed_max_length
    
    # Dynamic batching
    max_tokens = 65536  # Equivalent to 32 * 2048
    dynamic_sampler = DynamicBatchSampler(lengths, max_tokens)
    dynamic_batches = list(dynamic_sampler)
    dynamic_total_tokens = sum(
        len(batch) * max(lengths[i] for i in batch) 
        for batch in dynamic_batches
    )
    
    # Length bucketing
    bucket_boundaries = [256, 512, 1024, 1536, 2048]
    bucket_sampler = LengthBucketSampler(lengths, fixed_batch_size, bucket_boundaries)
    bucket_batches = list(bucket_sampler)
    bucket_total_tokens = sum(
        len(batch) * max(lengths[i] for i in batch)
        for batch in bucket_batches
    )
    
    print(f"Dynamic Batching Comparison")
    print(f"{'='*60}")
    print(f"Dataset: {len(lengths)} sequences")
    print(f"\nFixed Batch Size (32):")
    print(f"  Batches: {fixed_batches}")
    print(f"  Total tokens: {fixed_total_tokens:,}")
    print(f"  Avg tokens/batch: {fixed_total_tokens/fixed_batches:,.0f}")
    print(f"\nDynamic Batching:")
    print(f"  Batches: {len(dynamic_batches)}")
    print(f"  Total tokens: {dynamic_total_tokens:,}")
    print(f"  Avg tokens/batch: {dynamic_total_tokens/len(dynamic_batches):,.0f}")
    print(f"  Efficiency gain: {fixed_total_tokens/dynamic_total_tokens:.2f}x")
    print(f"\nLength Bucketing:")
    print(f"  Batches: {len(bucket_batches)}")
    print(f"  Total tokens: {bucket_total_tokens:,}")
    print(f"  Avg tokens/batch: {bucket_total_tokens/len(bucket_batches):,.0f}")
    print(f"  Efficiency gain: {fixed_total_tokens/bucket_total_tokens:.2f}x")
    
    # Visualize batch sizes
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Dynamic batch sizes
    dynamic_sizes = [len(batch) for batch in dynamic_batches]
    axes[0].hist(dynamic_sizes, bins=20, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Batch Size', fontsize=12)
    axes[0].set_ylabel('Count', fontsize=12)
    axes[0].set_title('Dynamic Batching', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    
    # Max length per batch
    dynamic_max_lengths = [max(lengths[i] for i in batch) for batch in dynamic_batches]
    axes[1].scatter(dynamic_sizes, dynamic_max_lengths, alpha=0.6)
    axes[1].set_xlabel('Batch Size', fontsize=12)
    axes[1].set_ylabel('Max Sequence Length', fontsize=12)
    axes[1].set_title('Batch Size vs Max Length', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    # Tokens per batch
    dynamic_tokens = [len(batch) * max(lengths[i] for i in batch) for batch in dynamic_batches]
    axes[2].hist(dynamic_tokens, bins=20, edgecolor='black', alpha=0.7, color='green')
    axes[2].axvline(max_tokens, color='red', linestyle='--', linewidth=2, label='Target')
    axes[2].set_xlabel('Total Tokens', fontsize=12)
    axes[2].set_ylabel('Count', fontsize=12)
    axes[2].set_title('Tokens per Batch', fontsize=14, fontweight='bold')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('dynamic_batching_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

benchmark_dynamic_batching()

## 3. Streaming Datasets (Petabyte Scale)

**Problem**: Datasets too large for disk/RAM
- GPT-3: 300B tokens
- LLaMA: 1.4T tokens
- Modern models: Multi-trillion tokens

**Solution**: Stream from cloud storage, process on-the-fly

In [ ]:
class StreamingDataset(IterableDataset):
    """Stream data from multiple sources without loading into memory.
    
    Simulates HuggingFace datasets in streaming mode.
    """
    
    def __init__(self, file_paths: List[str], buffer_size: int = 10000):
        super().__init__()
        self.file_paths = file_paths
        self.buffer_size = buffer_size
    
    def __iter__(self) -> Iterator[Dict[str, torch.Tensor]]:
        """Stream samples from files."""
        worker_info = torch.utils.data.get_worker_info()
        
        # Shard files across workers
        if worker_info is not None:
            worker_id = worker_info.id
            num_workers = worker_info.num_workers
            file_paths = self.file_paths[worker_id::num_workers]
        else:
            file_paths = self.file_paths
        
        # Stream from each file
        for file_path in file_paths:
            # In practice, this would read from S3/GCS/etc.
            # Here we simulate with random data
            for _ in range(self.buffer_size):
                # Simulate reading and tokenizing
                length = np.random.randint(100, 512)
                yield {
                    'input_ids': torch.randint(0, 32000, (length,)),
                    'attention_mask': torch.ones(length, dtype=torch.long),
                }


class InterleaveDataset(IterableDataset):
    """Interleave multiple streaming datasets.
    
    Used to mix different data sources (e.g., web, books, code)
    with specified sampling weights.
    """
    
    def __init__(self, datasets: List[IterableDataset], 
                 weights: List[float] = None, seed: int = 42):
        super().__init__()
        self.datasets = datasets
        self.weights = weights or [1.0] * len(datasets)
        self.weights = np.array(self.weights) / sum(self.weights)
        self.seed = seed
        self.rng = np.random.RandomState(seed)
    
    def __iter__(self) -> Iterator[Dict[str, torch.Tensor]]:
        """Interleave samples from datasets according to weights."""
        iterators = [iter(ds) for ds in self.datasets]
        
        while iterators:
            # Sample dataset according to weights
            idx = self.rng.choice(len(iterators), p=self.weights[:len(iterators)])
            
            try:
                sample = next(iterators[idx])
                yield sample
            except StopIteration:
                # Remove exhausted iterator
                iterators.pop(idx)
                self.weights = np.delete(self.weights, idx)
                if len(self.weights) > 0:
                    self.weights /= self.weights.sum()


# Demonstrate streaming
def demo_streaming():
    """Show streaming dataset in action."""
    
    print("Streaming Dataset Demo")
    print("="*60)
    
    # Create streaming datasets for different data sources
    web_data = StreamingDataset([f"web_shard_{i}.jsonl" for i in range(3)], buffer_size=100)
    book_data = StreamingDataset([f"books_shard_{i}.jsonl" for i in range(2)], buffer_size=100)
    code_data = StreamingDataset([f"code_shard_{i}.jsonl" for i in range(2)], buffer_size=100)
    
    # Interleave with custom weights (e.g., 60% web, 25% books, 15% code)
    dataset = InterleaveDataset(
        [web_data, book_data, code_data],
        weights=[0.6, 0.25, 0.15]
    )
    
    # Create dataloader with multiple workers for prefetching
    dataloader = DataLoader(
        dataset,
        batch_size=None,  # Dataset yields single samples
        num_workers=4,
        prefetch_factor=2,
    )
    
    # Measure streaming throughput
    start_time = time.time()
    samples_processed = 0
    tokens_processed = 0
    
    for i, batch in enumerate(dataloader):
        if i >= 1000:  # Process 1000 samples
            break
        samples_processed += 1
        tokens_processed += len(batch['input_ids'])
    
    elapsed = time.time() - start_time
    
    print(f"\nStreaming Performance:")
    print(f"  Samples: {samples_processed}")
    print(f"  Tokens: {tokens_processed:,}")
    print(f"  Time: {elapsed:.2f}s")
    print(f"  Throughput: {samples_processed/elapsed:.1f} samples/sec")
    print(f"  Throughput: {tokens_processed/elapsed:,.0f} tokens/sec")
    print(f"\nKey benefits:")
    print(f"  - No disk space needed (streams from cloud)")
    print(f"  - No RAM for full dataset")
    print(f"  - Can train on infinite data")
    print(f"  - Multi-worker prefetching hides latency")

demo_streaming()

## 4. DataLoader Optimization

**Key optimizations**:
1. **num_workers**: Multi-process data loading
2. **prefetch_factor**: How many batches to prepare ahead
3. **pin_memory**: Faster GPU transfer
4. **persistent_workers**: Keep workers alive between epochs

In [ ]:
class DummyDataset(Dataset):
    """Dummy dataset for benchmarking."""
    
    def __init__(self, size: int = 10000, seq_length: int = 512):
        self.size = size
        self.seq_length = seq_length
    
    def __len__(self) -> int:
        return self.size
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        # Simulate some processing time
        time.sleep(0.001)  # 1ms per sample
        return {
            'input_ids': torch.randint(0, 32000, (self.seq_length,)),
            'attention_mask': torch.ones(self.seq_length, dtype=torch.long),
        }


def benchmark_dataloader_configs():
    """Benchmark different DataLoader configurations."""
    
    dataset = DummyDataset(size=1000)
    batch_size = 32
    
    configs = [
        {'num_workers': 0, 'prefetch_factor': None, 'pin_memory': False, 'persistent_workers': False},
        {'num_workers': 2, 'prefetch_factor': 2, 'pin_memory': False, 'persistent_workers': False},
        {'num_workers': 4, 'prefetch_factor': 2, 'pin_memory': False, 'persistent_workers': False},
        {'num_workers': 4, 'prefetch_factor': 4, 'pin_memory': False, 'persistent_workers': False},
        {'num_workers': 4, 'prefetch_factor': 4, 'pin_memory': True, 'persistent_workers': True},
    ]
    
    results = []
    
    print("DataLoader Configuration Benchmark")
    print("="*80)
    
    for config in configs:
        # Handle prefetch_factor for num_workers=0
        loader_config = config.copy()
        if loader_config['num_workers'] == 0:
            loader_config.pop('prefetch_factor')
            loader_config.pop('persistent_workers')
        
        dataloader = DataLoader(
            dataset,
            batch_size=batch_size,
            **loader_config
        )
        
        # Benchmark
        start_time = time.time()
        batches = 0
        for batch in dataloader:
            batches += 1
            if batches >= 50:  # Process 50 batches
                break
        elapsed = time.time() - start_time
        
        throughput = (batches * batch_size) / elapsed
        results.append((config, throughput))
        
        config_str = ", ".join(f"{k}={v}" for k, v in config.items() if v is not None)
        print(f"{config_str:60s} | {throughput:6.1f} samples/sec")
    
    # Plot results
    labels = []
    throughputs = []
    for config, throughput in results:
        label = f"workers={config['num_workers']}"
        if config['prefetch_factor'] is not None:
            label += f", prefetch={config['prefetch_factor']}"
        if config['pin_memory']:
            label += ", pinned"
        if config['persistent_workers']:
            label += ", persistent"
        labels.append(label)
        throughputs.append(throughput)
    
    plt.figure(figsize=(12, 6))
    bars = plt.bar(range(len(throughputs)), throughputs, edgecolor='black')
    plt.xticks(range(len(labels)), labels, rotation=45, ha='right')
    plt.ylabel('Throughput (samples/sec)', fontsize=12)
    plt.title('DataLoader Configuration Benchmark', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig('dataloader_benchmark.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nSpeedup from optimal config: {max(throughputs)/min(throughputs):.2f}x")

benchmark_dataloader_configs()

## 5. Memory-Mapped Files

**Problem**: Fast random access to huge datasets
**Solution**: Memory-mapped files (mmap) - OS handles paging

In [ ]:
import mmap
import struct
import os

class MemoryMappedDataset(Dataset):
    """Dataset backed by memory-mapped file for fast random access.
    
    Used by HuggingFace datasets and modern training frameworks.
    Format: [num_samples (8 bytes)] + [offset_1 (8 bytes), offset_2, ...] + [data]
    """
    
    def __init__(self, file_path: str):
        self.file_path = file_path
        
        # Open file in memory-mapped mode
        self.file = open(file_path, 'r+b')
        self.mmap = mmap.mmap(self.file.fileno(), 0)
        
        # Read metadata
        self.num_samples = struct.unpack('Q', self.mmap[:8])[0]
        
        # Read offsets
        offset_bytes = self.num_samples * 8
        self.offsets = struct.unpack(
            f'{self.num_samples}Q',
            self.mmap[8:8+offset_bytes]
        )
        
        self.data_start = 8 + offset_bytes
    
    def __len__(self) -> int:
        return self.num_samples
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        # Get sample boundaries
        start = self.data_start + self.offsets[idx]
        end = self.data_start + self.offsets[idx + 1] if idx + 1 < self.num_samples else len(self.mmap)
        
        # Read sample data (already in memory via mmap)
        length = (end - start) // 4  # Assuming 4-byte integers
        data = struct.unpack(f'{length}I', self.mmap[start:end])
        
        return {
            'input_ids': torch.tensor(data, dtype=torch.long),
            'attention_mask': torch.ones(length, dtype=torch.long),
        }
    
    def __del__(self):
        """Clean up resources."""
        if hasattr(self, 'mmap'):
            self.mmap.close()
        if hasattr(self, 'file'):
            self.file.close()


def create_mmap_dataset(file_path: str, num_samples: int = 10000):
    """Create a memory-mapped dataset file."""
    
    with open(file_path, 'wb') as f:
        # Write number of samples
        f.write(struct.pack('Q', num_samples))
        
        # Reserve space for offsets
        offset_section_size = num_samples * 8
        f.write(b'\x00' * offset_section_size)
        
        # Write samples and record offsets
        offsets = []
        current_offset = 0
        
        for i in range(num_samples):
            offsets.append(current_offset)
            
            # Generate variable-length sample
            length = np.random.randint(100, 512)
            data = np.random.randint(0, 32000, length, dtype=np.uint32)
            
            # Write sample
            f.write(data.tobytes())
            current_offset += length * 4
        
        # Write offsets at the beginning
        f.seek(8)  # After num_samples
        f.write(struct.pack(f'{num_samples}Q', *offsets))
    
    print(f"Created memory-mapped dataset: {file_path}")
    print(f"  Samples: {num_samples:,}")
    print(f"  File size: {os.path.getsize(file_path) / 1024 / 1024:.2f} MB")


# Benchmark mmap vs regular file I/O
def benchmark_mmap():
    """Compare memory-mapped vs regular file access."""
    
    file_path = '/tmp/mmap_dataset.bin'
    create_mmap_dataset(file_path, num_samples=10000)
    
    dataset = MemoryMappedDataset(file_path)
    
    # Random access benchmark
    num_accesses = 1000
    indices = np.random.randint(0, len(dataset), num_accesses)
    
    start_time = time.time()
    for idx in indices:
        sample = dataset[idx]
    elapsed = time.time() - start_time
    
    print(f"\nMemory-Mapped Random Access Benchmark")
    print(f"="*60)
    print(f"Accesses: {num_accesses}")
    print(f"Time: {elapsed:.3f}s")
    print(f"Throughput: {num_accesses/elapsed:.1f} accesses/sec")
    print(f"\nKey benefits:")
    print(f"  - O(1) random access")
    print(f"  - OS handles caching automatically")
    print(f"  - Multiple processes can share same data")
    print(f"  - No need to load full dataset into RAM")
    
    # Cleanup
    del dataset
    os.remove(file_path)

benchmark_mmap()

## 6. Complete Training Pipeline

Putting it all together: streaming + packing + dynamic batching

In [ ]:
class OptimizedDataPipeline:
    """Production-ready data pipeline for LLM training."""
    
    def __init__(self, max_length: int = 2048, max_tokens_per_batch: int = 65536):
        self.max_length = max_length
        self.max_tokens_per_batch = max_tokens_per_batch
        self.packer = SequencePacker(max_length)
    
    def create_dataloader(
        self,
        dataset: IterableDataset,
        num_workers: int = 4,
        prefetch_factor: int = 4,
    ) -> DataLoader:
        """Create optimized dataloader."""
        
        def collate_fn(batch):
            """Pack and pad batch efficiently."""
            # Extract sequences
            sequences = [item['input_ids'].tolist() for item in batch]
            
            # Pack sequences
            packed = self.packer.pack_sequences(sequences)
            
            # Stack into tensors
            return {
                'input_ids': torch.stack([p['input_ids'] for p in packed]),
                'attention_mask': torch.stack([p['attention_mask'] for p in packed]),
                'position_ids': torch.stack([p['position_ids'] for p in packed]),
            }
        
        return DataLoader(
            dataset,
            batch_size=None,  # Handle batching in collate_fn
            num_workers=num_workers,
            prefetch_factor=prefetch_factor,
            pin_memory=True,
            persistent_workers=True if num_workers > 0 else False,
            collate_fn=None,  # Process individual samples
        )


# Full pipeline demo
def demo_full_pipeline():
    """Demonstrate complete optimized pipeline."""
    
    print("Complete Optimized Data Pipeline")
    print("="*80)
    
    # Create streaming dataset
    dataset = StreamingDataset([f"shard_{i}.jsonl" for i in range(5)], buffer_size=1000)
    
    # Create optimized pipeline
    pipeline = OptimizedDataPipeline(max_length=2048)
    dataloader = pipeline.create_dataloader(dataset, num_workers=4, prefetch_factor=4)
    
    # Measure throughput
    start_time = time.time()
    total_tokens = 0
    batches_processed = 0
    
    for i, batch in enumerate(dataloader):
        if i >= 100:  # Process 100 batches
            break
        
        # Count real tokens (non-padding)
        real_tokens = batch['attention_mask'].sum().item()
        total_tokens += real_tokens
        batches_processed += 1
    
    elapsed = time.time() - start_time
    
    print(f"\nPipeline Performance:")
    print(f"  Batches: {batches_processed}")
    print(f"  Tokens: {total_tokens:,}")
    print(f"  Time: {elapsed:.2f}s")
    print(f"  Throughput: {total_tokens/elapsed:,.0f} tokens/sec")
    print(f"\nOptimizations Applied:")
    print(f"  [x] Sequence packing (2-3x speedup)")
    print(f"  [x] Streaming (infinite data)")
    print(f"  [x] Multi-worker prefetching (hide latency)")
    print(f"  [x] Pin memory (faster GPU transfer)")
    print(f"  [x] Persistent workers (avoid startup cost)")

demo_full_pipeline()

## Key Takeaways

### Sequence Packing
- **2-3x throughput gain** by eliminating padding waste
- Critical for training efficiency
- Used by all modern LLM training pipelines

### Dynamic Batching
- Adjust batch size based on sequence length
- Better GPU utilization
- Length bucketing reduces padding within batches

### Streaming
- Train on datasets larger than disk/RAM
- Essential for trillion-token training
- HuggingFace datasets library makes this easy

### DataLoader Optimization
- **num_workers=4-8**: Good balance for most systems
- **prefetch_factor=4**: Keep data ready
- **pin_memory=True**: Faster GPU transfer
- **persistent_workers=True**: Avoid worker startup cost

### Memory-Mapped Files
- Fast random access to huge datasets
- OS handles caching automatically
- Multiple processes can share data

### Production Best Practices
1. Use sequence packing (always)
2. Stream from cloud storage
3. Multi-worker prefetching (4-8 workers)
4. Length bucketing to reduce padding
5. Monitor data loading time vs training time
6. Target: data loading < 10% of total time